# Laboratorium 3: Logika symboliczna i systemy regułowe

Dane: fragment ontologii uczelni z laboratorium 1 (tekst Turtle w notebooku) oraz baza reguł diagnostycznych sieci komputerowej. Cel: zbudować w czystym Pythonie unifikację, wnioskowanie w przód i wstecz oraz mały system ekspercki z objaśnieniami, a potem porównać je z regułami nad grafem RDF.

In [ ]:
# W Colabie brakuje rdflib i owlrl, reszta bibliotek jest preinstalowana.
%pip install -q rdflib owlrl

In [ ]:
import itertools
import time
from collections import defaultdict, namedtuple

import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd
import rdflib
import owlrl
from rdflib import Graph, Namespace, RDF, Literal

print("rdflib", rdflib.__version__, "| networkx", nx.__version__, "| pandas", pd.__version__)

## Dane wejściowe

Poniższe komórki są gotowe. Pierwsza zawiera fragment ontologii uczelni w Turtle, druga zamienia trójki na fakty w postaci krotek: `rdf:type` daje fakt jednoargumentowy, na przykład `("Student", "anna")`, a każda inna właściwość fakt dwuargumentowy, na przykład `("zaliczyl", "anna", "WDP")`. Etykiety `rdfs:label` pomijamy.

In [ ]:
TTL = """@prefix u:    <http://example.org/uczelnia#> .
@prefix rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix xsd:  <http://www.w3.org/2001/XMLSchema#> .

u:WI a u:Wydzial ; rdfs:label "Wydział Informatyki" .

u:WDP  a u:KursObowiazkowy ; rdfs:label "Wstęp do programowania" ; u:ects 6 ; u:naWydziale u:WI .
u:Log  a u:KursObowiazkowy ; rdfs:label "Logika dla informatyków" ; u:ects 4 ; u:naWydziale u:WI .
u:Stat a u:KursObowiazkowy ; rdfs:label "Statystyka" ; u:ects 5 ; u:naWydziale u:WI .
u:PO   a u:KursObowiazkowy ; rdfs:label "Programowanie obiektowe" ; u:ects 6 ; u:naWydziale u:WI ;
       u:wymagaKursu u:WDP .
u:BD   a u:KursObowiazkowy ; rdfs:label "Bazy danych" ; u:ects 5 ; u:naWydziale u:WI ;
       u:wymagaKursu u:WDP .
u:AiSD a u:KursObowiazkowy ; rdfs:label "Algorytmy i struktury danych" ; u:ects 6 ; u:naWydziale u:WI ;
       u:wymagaKursu u:PO .
u:SI   a u:KursObowiazkowy ; rdfs:label "Sztuczna inteligencja" ; u:ects 5 ; u:naWydziale u:WI ;
       u:wymagaKursu u:AiSD , u:Log .
u:UM   a u:KursObieralny ; rdfs:label "Uczenie maszynowe" ; u:ects 5 ; u:naWydziale u:WI ;
       u:wymagaKursu u:AiSD , u:Stat .
u:MRW  a u:KursObieralny ; rdfs:label "Metody reprezentacji wiedzy" ; u:ects 4 ; u:naWydziale u:WI ;
       u:wymagaKursu u:SI , u:BD .
u:GK   a u:KursObieralny ; rdfs:label "Grafika komputerowa" ; u:ects 4 ; u:naWydziale u:WI ;
       u:wymagaKursu u:PO .

u:kmazur      a u:Pracownik ; u:imie "Katarzyna" ; u:nazwisko "Mazur" ; u:rokUrodzenia 1979 ;
              u:naWydziale u:WI ; u:prowadzi u:SI , u:MRW .
u:lkrawczyk   a u:Pracownik ; u:imie "Leszek" ; u:nazwisko "Krawczyk" ; u:rokUrodzenia 1971 ;
              u:naWydziale u:WI ; u:prowadzi u:WDP , u:PO .
u:mpiotrowska a u:Pracownik ; u:imie "Monika" ; u:nazwisko "Piotrowska" ; u:rokUrodzenia 1985 ;
              u:naWydziale u:WI ; u:prowadzi u:AiSD , u:Log .
u:ngrabowski  a u:Pracownik ; u:imie "Norbert" ; u:nazwisko "Grabowski" ; u:rokUrodzenia 1968 ;
              u:naWydziale u:WI ; u:prowadzi u:BD , u:Stat , u:UM .
u:tsadowski   a u:Pracownik ; u:imie "Tomasz" ; u:nazwisko "Sadowski" ; u:rokUrodzenia 1990 ;
              u:naWydziale u:WI ; u:prowadzi u:GK .
u:opawlowska  a u:Pracownik ; u:imie "Olga" ; u:nazwisko "Pawłowska" ; u:rokUrodzenia 1982 ;
              u:naWydziale u:WI .

u:anna     a u:Student ; u:imie "Anna" ; u:nazwisko "Nowak" ; u:rokStudiow 1 ; u:rokUrodzenia 2005 ;
           u:zaliczyl u:WDP , u:Log ; u:zapisanyNa u:PO .
u:bartosz  a u:Student ; u:imie "Bartosz" ; u:nazwisko "Kowalski" ; u:rokStudiow 2 ; u:rokUrodzenia 2004 ;
           u:zaliczyl u:WDP , u:PO , u:Log , u:Stat ; u:zapisanyNa u:AiSD .
u:celina   a u:Student ; u:imie "Celina" ; u:nazwisko "Wiśniewska" ; u:rokStudiow 3 ; u:rokUrodzenia 2003 ;
           u:zaliczyl u:WDP , u:PO , u:AiSD , u:Log , u:Stat , u:BD ; u:zapisanyNa u:SI .
u:dawid    a u:Student ; u:imie "Dawid" ; u:nazwisko "Wójcik" ; u:rokStudiow 4 ; u:rokUrodzenia 2002 ;
           u:zaliczyl u:WDP , u:PO , u:AiSD , u:Log , u:Stat , u:BD , u:SI ; u:zapisanyNa u:MRW .
u:ewa      a u:Student ; u:imie "Ewa" ; u:nazwisko "Kamińska" ; u:rokStudiow 1 ; u:rokUrodzenia 2006 ;
           u:zapisanyNa u:WDP .
u:filip    a u:Student ; u:imie "Filip" ; u:nazwisko "Lewandowski" ; u:rokStudiow 2 ; u:rokUrodzenia 2004 ;
           u:zaliczyl u:WDP , u:PO , u:BD .
u:gabriela a u:Student ; u:imie "Gabriela" ; u:nazwisko "Zielińska" ; u:rokStudiow 4 ; u:rokUrodzenia 2001 ;
           u:zaliczyl u:WDP , u:PO , u:AiSD , u:Log , u:Stat , u:BD , u:SI , u:UM ; u:zapisanyNa u:MRW .
u:hubert   a u:Student ; u:imie "Hubert" ; u:nazwisko "Szymański" ; u:rokStudiow 4 ; u:rokUrodzenia 2002 ;
           u:zaliczyl u:WDP , u:PO , u:AiSD , u:BD , u:SI ; u:zapisanyNa u:MRW .
u:iga      a u:Student ; u:imie "Iga" ; u:nazwisko "Woźniak" ; u:rokStudiow 3 ; u:rokUrodzenia 2003 ;
           u:zaliczyl u:WDP , u:PO , u:Stat ; u:zapisanyNa u:AiSD , u:GK .
u:jan      a u:Student ; u:imie "Jan" ; u:nazwisko "Dąbrowski" ; u:rokStudiow 5 ; u:rokUrodzenia 2000 ;
           u:zaliczyl u:WDP , u:PO , u:AiSD , u:Log , u:Stat , u:BD , u:SI , u:MRW .

u:praca1 a u:PracaDyplomowa ; rdfs:label "Ontologia planu studiów" ; u:autor u:jan ; u:promotor u:kmazur .
u:praca2 a u:PracaDyplomowa ; rdfs:label "Prognozowanie obciążenia serwerów" ; u:autor u:gabriela ; u:promotor u:ngrabowski .
u:praca3 a u:PracaDyplomowa ; rdfs:label "Reguły dostępu w grafie wiedzy" ; u:autor u:dawid ; u:promotor u:kmazur .
"""

In [ ]:
U = Namespace("http://example.org/uczelnia#")

def local(node):
    """Nazwa lokalna IRI albo wartość literału."""
    if isinstance(node, Literal):
        return node.toPython()
    return str(node).split("#")[-1]

def ttl_to_facts(ttl):
    g = Graph()
    g.parse(data=ttl, format="turtle")
    facts = set()
    for s, p, o in g:
        if p == RDF.type:
            facts.add((local(o), local(s)))
        elif str(p).startswith(str(U)):
            facts.add((local(p), local(s), local(o)))
    return g, facts

g_uczelnia, FACTS = ttl_to_facts(TTL)
print(len(g_uczelnia), "trójek,", len(FACTS), "faktów")
print(sorted(f for f in FACTS if f[0] == "wymagaKursu"))

## Zadanie 1: Logika zdań i logika predykatów

### 1a. Formalizacja

Zapiszcie w logice predykatów (z predykatami ze słownictwa uczelni) zdania:

1. Każdy wykładowca jest pracownikiem.
2. Pracownik, który prowadzi jakiś kurs, jest wykładowcą.
3. Kto zaliczył kurs, zaliczył też każdy jego bezpośredni prerekwizyt.
4. Istnieje kurs, którego nie zaliczył żaden student.
5. Każda praca dyplomowa ma dokładnie jednego promotora.
6. Nie każdy student zapisany na kurs obieralny ma zaliczone wszystkie jego prerekwizyty.

Przy każdej formule zaznaczcie, czy da się ją zapisać jako klauzulę Horna (regułę z jednym pozytywnym wnioskiem i koniunkcją przesłanek).

(uzupełnij)

### 1b. Sprawdzenie formuł na danych

Fakty z ontologii wyznaczają skończony model. Sprawdźcie w nim zdania 3 i 4 (zwykłymi wyrażeniami Pythona po zbiorze `FACTS`). Dla zdania 3 wypiszcie kontrprzykłady.

In [ ]:
# Wyodrębnij z FACTS zbiory par zaliczyl i wymagaKursu oraz zbiory studentów i kursów.
# Sprawdź zdanie 3 (wypisz kontrprzykłady) i zdanie 4 (wypisz kurs będący świadkiem).

### 1c. Wynikanie w logice zdań przez tablicę prawdy

Formuły logiki zdań zapisujemy jako krotki: symbol to napis, a złożenia to `("not", f)`, `("and", f, g, ...)`, `("or", f, g, ...)`, `("imp", f, g)`, `("iff", f, g)`. Baza `KB_EWA` opisuje, co wiadomo o zaliczeniach jednej studentki.

In [ ]:
KB_EWA = [
    ("imp", "zal_AiSD", "zal_PO"),     # kto zaliczył AiSD, zaliczył PO
    ("imp", "zal_PO", "zal_WDP"),
    ("imp", "zal_BD", "zal_WDP"),
    ("or", "zal_AiSD", "zal_BD"),      # z protokołu wiadomo tylko, że zaliczyła AiSD albo BD
    ("imp", ("and", "zal_SI", ("not", "zal_Log")), "zgoda_dziekana"),
]
QUERIES = ["zal_WDP", "zal_PO", ("or", "zal_PO", "zal_BD"), ("imp", "zal_SI", "zal_Log")]

In [ ]:
# Napisz pl_true(f, model), symbols(f) i tt_entails(kb, alpha) zwracające
# (czy wynika, liczba sprawdzonych modeli, kontrmodel). Policz liczbę modeli KB_EWA
# i zbuduj tabelę wyników dla zapytań z QUERIES.

### 1d. Złożoność: tablica prawdy a DPLL

Baza łańcuchowa `p1, p1 -> p2, ..., p(n-1) -> pn` pozwala wywnioskować `pn`. Zmierzcie czas i liczbę modeli sprawdzanych przez tablicę prawdy dla rosnącego n. Następnie napiszcie DPLL na klauzulach (zbiory literałów, negacja jako `~p`) z propagacją jednostkową i sprawdźcie to samo wynikanie przez niespełnialność `KB and not pn`. Na koniec uruchomcie DPLL na zasadzie szufladkowej (n + 1 gołębi, n przegródek), której klauzule generuje gotowa funkcja `pigeonhole`.

In [ ]:
def chain_kb(n):
    kb = ["p1"] + [("imp", f"p{i}", f"p{i + 1}") for i in range(1, n)]
    return kb, f"p{n}"

def chain_cnf(n):
    """KB łańcuchowa w CNF razem z zanegowanym zapytaniem."""
    cl = [frozenset({"p1"})] + [frozenset({f"~p{i}", f"p{i + 1}"}) for i in range(1, n)]
    return cl + [frozenset({f"~p{n}"})]

def pigeonhole(n):
    """n + 1 gołębi, n przegródek: formuła niespełnialna."""
    v = lambda i, j: f"g{i}_{j}"
    cl = [frozenset(v(i, j) for j in range(n)) for i in range(n + 1)]
    for j in range(n):
        for i1, i2 in itertools.combinations(range(n + 1), 2):
            cl.append(frozenset({"~" + v(i1, j), "~" + v(i2, j)}))
    return cl

In [ ]:
# Dla n w [4, 8, 10, 12, 14, 16, 18] zmierz czas i liczbę modeli sprawdzonych przez tt_entails(chain_kb(n)).

In [ ]:
# Napisz dpll(clauses, stats) z propagacją jednostkową i rozgałęzieniem na literale,
# licz wywołania i propagacje. Sprawdź wynikanie dla chain_cnf(n), n w [4, 8, 18, 100, 1000].

In [ ]:
# Uruchom dpll na pigeonhole(n) dla n od 2 do 7, zapisz liczbę zmiennych, klauzul, wywołań i czas.

In [ ]:
# Narysuj na skali logarytmicznej liczbę modeli w funkcji n oraz liczbę wywołań DPLL dla zasady szufladkowej.

### Wnioski do zadania 1

(uzupełnij)

## Zadanie 2: Unifikacja

Termy: zmienna to napis zaczynający się od `?` (np. `"?s"`), stała to dowolny inny napis lub liczba, a term złożony (i atom) to krotka `(funktor, arg1, arg2, ...)`. Podstawienie to słownik zmienna -> term, w którym wiązania mogą być łańcuchowe (`?x -> ?y -> anna`).

In [ ]:
# Napisz is_var, walk (podążanie za łańcuchem wiązań), occurs (occurs check),
# unify(x, y, theta=None, occurs_check=True) zwracające MGU albo None,
# subst(t, theta) stosujące podstawienie oraz funkcje show(t) i show_theta(theta) do czytelnego wypisywania.

In [ ]:
UNIFY_TESTS = [
    (("zaliczyl", "?s", "SI"), ("zaliczyl", "anna", "?k")),
    (("wymagaKursu", "?x", "?x"), ("wymagaKursu", "MRW", "SI")),
    (("p", "?x", ("f", "?x")), ("p", ("g", "?y"), ("f", ("g", "a")))),
    (("zna", "jan", "?x"), ("zna", "?y", ("matka", "?y"))),
    (("zna", "jan", "?x"), ("zna", "?x", "ewa")),
    (("zna", "jan", "?x"), ("zna", "?x2", "ewa")),
    ("?x", ("f", "?x")),
    (("p", "?x", "?y", "?z"), ("p", "?y", "?z", "b")),
    (("p", "a"), ("p", "a", "b")),
]

In [ ]:
# Dla każdej pary z UNIFY_TESTS wypisz MGU (albo porażkę). Dla pary ("?x", f(?x))
# porównaj wynik z occurs check i bez niego.

In [ ]:
# Zastosuj podstawienie uzyskane bez occurs check do zmiennej ?x i pokaż, co się dzieje.

### 2b. Koszt occurs check

Unifikacja `p(?x1, ..., ?xn)` z `p(f(?x0, ?x0), f(?x1, ?x1), ..., f(?x(n-1), ?x(n-1)))` wiąże każdą zmienną z termem dwa razy większym od poprzedniego. Zmierzcie czas unifikacji z occurs check i bez niego oraz rozmiar termu `subst("?xn", theta)` (liczba węzłów) dla n od 10 do 18.

In [ ]:
# Zbuduj oba termy dla n w [10, 12, 14, 16, 18], zmierz czas unify z occurs check i bez,
# policz liczbę wiązań i rozmiar termu subst("?xn", theta).

### Wnioski do zadania 2

(uzupełnij)

## Zadanie 3: Wnioskowanie w przód

Reguła ma nazwę, głowę (atom) i ciało (listę literałów). Literał negatywny zapisujemy jako `("not", atom)` i rozumiemy jako negację przez porażkę: jest spełniony, gdy podstawionego atomu nie ma w bazie faktów. Reguły R1 do R4 odtwarzają hierarchię klas z ontologii, R5 definicję wykładowcy, R6 i R7 przechodnie domknięcie prerekwizytów.

In [ ]:
Rule = namedtuple("Rule", "name head body")

def NOT(atom):
    return ("not", atom)

RULES = [
    Rule("R1", ("Kurs", "?k"), [("KursObowiazkowy", "?k")]),
    Rule("R2", ("Kurs", "?k"), [("KursObieralny", "?k")]),
    Rule("R3", ("Osoba", "?x"), [("Student", "?x")]),
    Rule("R4", ("Osoba", "?x"), [("Pracownik", "?x")]),
    Rule("R5", ("Wykladowca", "?p"), [("Pracownik", "?p"), ("prowadzi", "?p", "?k")]),
    Rule("R6", ("wymagaPosrednio", "?a", "?b"), [("wymagaKursu", "?a", "?b")]),
    Rule("R7", ("wymagaPosrednio", "?a", "?c"), [("wymagaKursu", "?a", "?b"), ("wymagaPosrednio", "?b", "?c")]),
    Rule("R8", ("promotorStudenta", "?s", "?p"), [("autor", "?t", "?s"), ("promotor", "?t", "?p")]),
    Rule("R9", ("brakujePrereq", "?s", "?k"),
         [("Student", "?s"), ("wymagaPosrednio", "?k", "?w"), NOT(("zaliczyl", "?s", "?w"))]),
    Rule("R10", ("mozeSieZapisac", "?s", "?k"),
         [("Student", "?s"), ("Kurs", "?k"), NOT(("zaliczyl", "?s", "?k")), NOT(("brakujePrereq", "?s", "?k"))]),
    Rule("R11", ("naruszenie", "?s", "?k"), [("zapisanyNa", "?s", "?k"), ("brakujePrereq", "?s", "?k")]),
]
HORN = [r for r in RULES if r.name in {"R1", "R2", "R3", "R4", "R5", "R6", "R7", "R8"}]

def show_rule(r):
    return f"{r.name}: {show(r.head)} :- " + ", ".join(
        ("not " + show(l[1])) if l[0] == "not" else show(l) for l in r.body)

for r in RULES:
    print(show_rule(r))

### 3a. Punkt stały dla reguł Horna

Napiszcie `match_body(body, index, theta)` (generator podstawień spełniających ciało; `index` to słownik predykat -> lista faktów) i `forward_chain(rules, facts)`, który w każdej iteracji dopasowuje wszystkie reguły do faktów znanych na początku iteracji, dopisuje nowe fakty na końcu iteracji i kończy, gdy nic nowego nie powstaje. Zbierzcie dla każdej iteracji liczbę odpaleń reguł (podstawień spełniających ciało) i liczbę nowych faktów. Uruchomcie silnik na regułach `HORN`.

In [ ]:
# Napisz build_index, match_body (z obsługą literału ("not", atom) jako negacji przez porażkę)
# i forward_chain(rules, facts) zwracające zbiór faktów i tabelę historii iteracji.
# Uruchom na HORN i wypisz liczbę faktów przed i po.

In [ ]:
# Wypisz wywnioskowane prerekwizyty MRW, listę wykładowców i pary promotorStudenta.

### 3b. Negacja i stratyfikacja

Uruchomcie ten sam silnik na wszystkich regułach `RULES` naraz. Potem zbudujcie graf zależności predykatów (krawędź z predykatu w ciele do predykatu głowy, oznaczona jako negatywna, gdy literał jest zanegowany), przypiszcie predykatom warstwy (predykat z głowy jest w warstwie co najmniej takiej jak predykaty z ciała, a ściśle wyższej niż predykaty zanegowane) i wykonajcie wnioskowanie warstwa po warstwie. Porównajcie fakty `mozeSieZapisac` z obu przebiegów.

In [ ]:
# 1. Uruchom forward_chain na wszystkich RULES i pokaż historię iteracji.
# 2. Zbuduj graf zależności predykatów (networkx.DiGraph, atrybut krawędzi neg),
#    napisz stratify(rules) zwracające słownik predykat -> warstwa (z wykrywaniem cyklu przez negację)
#    i stratified_chain(rules, facts) wykonujące forward_chain kolejno dla każdej warstwy.

In [ ]:
# Porównaj zbiory mozeSieZapisac z obu przebiegów: ile jest fałszywych wniosków w przebiegu naiwnym,
# ile par jest jednocześnie w mozeSieZapisac i brakujePrereq. Wypisz naruszenia i tabelę "kto, na co".

### 3c. Niemonotoniczność i założenie świata zamkniętego

Sprawdźcie, co dzieje się z wnioskami po dopisaniu faktów: (a) `("zaliczyl", "hubert", "Log")`, (b) `("wymagaKursu", "MRW", "UM")`. Policzcie wnioski, które zniknęły, i te, które się pojawiły. Na koniec porównajcie wynik reguły R9 z wariantem, w którym brak prerekwizytu sprawdza się tylko dla bezpośrednich wymagań (`wymagaKursu` zamiast `wymagaPosrednio`).

In [ ]:
# Napisz funkcję, która dodaje jeden fakt, przelicza stratified_chain i wypisuje,
# które fakty mozeSieZapisac, brakujePrereq i naruszenie zniknęły, a które się pojawiły.

In [ ]:
# Zamień w R9 wymagaPosrednio na wymagaKursu, przelicz i pokaż różnice w mozeSieZapisac i naruszenie.

### Wnioski do zadania 3

(uzupełnij)

## Zadanie 4: Wnioskowanie wstecz

Napiszcie generator `prove(goal, theta, depth)`, który przeszukuje w głąb (jak Prolog): najpierw dopasowuje cel do faktów, potem do głów reguł w kolejności z listy, a dla każdej reguły dowodzi kolejnych literałów ciała. Przed użyciem reguły przemianujcie jej zmienne (standaryzacja rozdzielająca). Cel negatywny `("not", atom)` udowadniacie, gdy próba dowodu atomu się nie powiedzie. Generator ma zwracać pary (podstawienie, drzewo dowodu), a w słowniku `stats` liczyć użycia reguł (udane unifikacje z głową) i dopasowania do faktów. Przekroczenie maksymalnej głębokości ma zgłaszać wyjątek.

In [ ]:
# Napisz klasę Prover z metodami rename(rule), prove(goal, theta, depth) (generator par: podstawienie, drzewo),
# prove_all(goals, theta, depth) i ask(goal), licznikiem stats oraz wyjątkiem DepthLimit.
# Drzewo dowodu: krotka (cel, "fakt" | nazwa reguły | "negacja jako porażka", lista poddrzew).
# Napisz print_tree(tree, theta) wypisujące drzewo z wcięciami.

In [ ]:
# Zadaj zapytanie wymagaPosrednio(MRW, ?x): wypisz odpowiedzi, liczbę odpowiedzi i różnych odpowiedzi,
# statystyki oraz drzewo dowodu jednej z nich.

In [ ]:
# Udowodnij mozeSieZapisac(dawid, MRW) i znajdź wszystkie naruszenia; wypisz statystyki i drzewa dowodu.

### 4b. Porównanie z wnioskowaniem w przód

Dla kilku zapytań porównajcie liczbę użyć reguł we wnioskowaniu wstecz z liczbą odpaleń reguł w pełnym przebiegu warstwowym w przód (suma po warstwach i iteracjach).

In [ ]:
# Policz łączną liczbę odpaleń w przebiegu warstwowym w przód. Dla zapytań:
# wymagaPosrednio(MRW, ?x), Wykladowca(kmazur), mozeSieZapisac(dawid, MRW), mozeSieZapisac(hubert, ?k),
# mozeSieZapisac(?s, ?k), naruszenie(?s, ?k) zbuduj tabelę: liczba różnych odpowiedzi, użycia reguł,
# dopasowania faktów, czas.

### 4c. Reguła lewostronnie rekurencyjna

Zamieńcie R7 na logicznie równoważną regułę `wymagaPosrednio(?a, ?c) :- wymagaPosrednio(?a, ?b), wymagaKursu(?b, ?c)`. Sprawdźcie wynik wnioskowania w przód i wstecz.

In [ ]:
# Zbuduj RULES_LEFT z regułą R7L, sprawdź wnioskowanie w przód (czy wynik się nie zmienił),
# a we wnioskowaniu wstecz pobierz dwie pierwsze odpowiedzi (itertools.islice), a potem spróbuj pobrać resztę.

### Wnioski do zadania 4

(uzupełnij)

## Zadanie 5: Reguły nad grafem RDF

Te same reguły uczelni zapiszcie jako zapytania SPARQL CONSTRUCT nad grafem `g_uczelnia`. Zapytania z jednej warstwy wykonujcie w pętli, dopisując wynik do grafu, aż liczba trójek przestanie rosnąć. Negację wyraźcie przez `FILTER NOT EXISTS`. Porównajcie wywnioskowane trójki z faktami z zadania 3. Na koniec sprawdźcie, ile z tego potrafi wywnioskować reasoner OWL 2 RL (`owlrl`), gdy `u:wymagaKursu` oznaczymy jako `owl:TransitiveProperty`.

In [ ]:
# Zapisz reguły R1 do R11 jako zapytania CONSTRUCT pogrupowane w warstwy (lista list),
# napisz sparql_fixpoint(graph, strata) dopisujące wyniki do kopii grafu do punktu stałego w każdej warstwie
# (wypisz liczbę rund i trójek) i zmierz czas.

In [ ]:
# Zamień wynikowy graf z powrotem na fakty (ttl_to_facts na serializacji) i zbuduj tabelę porównania
# liczby faktów dla każdego wywnioskowanego predykatu (silnik w Pythonie i SPARQL) oraz zgodności zbiorów.

In [ ]:
# Policz pary osiągalne ścieżką u:wymagaKursu+ w SPARQL. Dodaj do kopii grafu owl:TransitiveProperty
# dla u:wymagaKursu i podklasy kursów, uruchom owlrl.DeductiveClosure(owlrl.OWLRL_Semantics)
# i porównaj domknięcie wymagaKursu z wymagaPosrednio z zadania 3.

### Wnioski do zadania 5

(uzupełnij)

## Zadanie 6: System ekspercki z objaśnieniami

Dziedzina: komputer nie ma dostępu do internetu. Fakty mają postać par (atrybut, wartość). Reguła ma warunki (lista par) i jeden wniosek. Atrybuty z `ASKABLE` są obserwacjami, o które system może zapytać użytkownika. Odpowiedzi użytkownika symulują słowniki w `SCENARIOS` (kolejność kluczy to kolejność, w jakiej użytkownik zgłaszał obserwacje). Brak klucza oznacza odpowiedź "nie wiem".

In [ ]:
ESRule = namedtuple("ESRule", "name conditions conclusion")

ES_RULES = [
    ESRule("R01", [("polaczenie", "przewodowe"), ("dioda_karty", "nie")], ("warstwa1", "awaria")),
    ESRule("R02", [("polaczenie", "wifi"), ("wifi_polaczone", "nie")], ("warstwa1", "awaria")),
    ESRule("R03", [("warstwa1", "awaria"), ("polaczenie", "przewodowe")], ("diagnoza", "kabel_lub_port")),
    ESRule("R04", [("warstwa1", "awaria"), ("polaczenie", "wifi")], ("diagnoza", "polacz_z_wifi")),
    ESRule("R05", [("adres_ip", "apipa")], ("dhcp", "awaria")),
    ESRule("R06", [("adres_ip", "brak")], ("dhcp", "awaria")),
    ESRule("R07", [("dhcp", "awaria"), ("inne_urzadzenia", "tak")], ("diagnoza", "odnow_dzierzawe_dhcp")),
    ESRule("R08", [("dhcp", "awaria"), ("inne_urzadzenia", "nie")], ("diagnoza", "restart_routera")),
    ESRule("R09", [("adres_ip", "poprawny"), ("ping_brama", "nie")], ("lan", "awaria")),
    ESRule("R10", [("lan", "awaria"), ("inne_urzadzenia", "tak")], ("diagnoza", "zapora_lub_sterownik")),
    ESRule("R11", [("lan", "awaria"), ("inne_urzadzenia", "nie")], ("diagnoza", "restart_routera")),
    ESRule("R12", [("ping_brama", "tak"), ("ping_zewnetrzny", "nie")], ("wan", "awaria")),
    ESRule("R13", [("wan", "awaria"), ("dioda_wan", "nie")], ("diagnoza", "awaria_u_dostawcy")),
    ESRule("R14", [("wan", "awaria"), ("dioda_wan", "tak")], ("diagnoza", "restart_routera")),
    ESRule("R15", [("ping_zewnetrzny", "tak"), ("dns", "nie")], ("diagnoza", "zmien_serwer_dns")),
    ESRule("R16", [("ping_zewnetrzny", "tak"), ("dns", "tak")], ("siec", "sprawna")),
    ESRule("R17", [("siec", "sprawna")], ("diagnoza", "problem_aplikacji")),
    ESRule("R18", [("siec", "sprawna"), ("proxy", "tak")], ("diagnoza", "wylacz_proxy")),
    ESRule("R19", [("siec", "sprawna"), ("blad_certyfikatu", "tak")], ("podejrzenie", "czas_lub_przechwycenie")),
    ESRule("R20", [("podejrzenie", "czas_lub_przechwycenie"), ("zegar_poprawny", "nie")], ("diagnoza", "ustaw_zegar")),
    ESRule("R21", [("podejrzenie", "czas_lub_przechwycenie"), ("zegar_poprawny", "tak")],
           ("diagnoza", "sprawdz_antywirus_https")),
    ESRule("R22", [("siec", "sprawna"), ("vpn", "tak")], ("diagnoza", "sprawdz_vpn")),
    ESRule("R23", [("ping_brama", "tak"), ("ping_zewnetrzny", "nie"), ("vpn", "tak")], ("diagnoza", "sprawdz_vpn")),
    ESRule("R24", [("dhcp", "awaria"), ("polaczenie", "wifi"), ("inne_urzadzenia", "tak")],
           ("diagnoza", "zapomnij_siec_wifi")),
]

ASKABLE = {
    "polaczenie": "Komputer łączy się kablem czy przez Wi-Fi?",
    "dioda_karty": "Czy świeci dioda przy gnieździe sieciowym?",
    "wifi_polaczone": "Czy system pokazuje połączenie z siecią Wi-Fi?",
    "adres_ip": "Jaki adres IPv4 pokazuje ipconfig (poprawny, apipa 169.254.x.x, brak)?",
    "ping_brama": "Czy odpowiada ping do bramy domyślnej?",
    "ping_zewnetrzny": "Czy odpowiada ping do 1.1.1.1?",
    "dns": "Czy nslookup rozwiązuje nazwę example.org?",
    "inne_urzadzenia": "Czy inne urządzenia w tej sieci mają internet?",
    "dioda_wan": "Czy świeci dioda WAN/Internet na routerze?",
    "proxy": "Czy w systemie ustawiono serwer proxy?",
    "blad_certyfikatu": "Czy przeglądarka zgłasza błędy certyfikatów?",
    "zegar_poprawny": "Czy data i godzina w systemie są poprawne?",
    "vpn": "Czy działa klient VPN?",
}

SCENARIOS = {
    "S1 kabel": {"polaczenie": "przewodowe", "inne_urzadzenia": "tak", "dioda_karty": "nie",
                 "ping_brama": "nie", "adres_ip": "apipa"},
    "S2 proxy": {"polaczenie": "wifi", "wifi_polaczone": "tak", "adres_ip": "poprawny", "ping_brama": "tak",
                 "ping_zewnetrzny": "tak", "dns": "tak", "proxy": "tak"},
    "S3 dns": {"polaczenie": "wifi", "wifi_polaczone": "tak", "adres_ip": "poprawny", "ping_brama": "tak",
               "ping_zewnetrzny": "tak", "dns": "nie"},
    "S4 dostawca": {"polaczenie": "przewodowe", "dioda_karty": "tak", "adres_ip": "poprawny",
                    "ping_brama": "tak", "ping_zewnetrzny": "nie", "dioda_wan": "nie", "inne_urzadzenia": "nie"},
    "S5 wifi dhcp": {"polaczenie": "wifi", "wifi_polaczone": "tak", "inne_urzadzenia": "tak", "adres_ip": "apipa"},
    "S6 zegar": {"polaczenie": "przewodowe", "dioda_karty": "tak", "adres_ip": "poprawny", "ping_brama": "tak",
                 "ping_zewnetrzny": "tak", "dns": "tak", "blad_certyfikatu": "tak", "zegar_poprawny": "nie"},
    "S7 vpn": {"polaczenie": "wifi", "wifi_polaczone": "tak", "adres_ip": "poprawny", "ping_brama": "tak",
               "vpn": "tak", "ping_zewnetrzny": "nie", "dioda_wan": "tak"},
}
print(len(ES_RULES), "reguł,", len(ASKABLE), "pytań,", len(SCENARIOS), "scenariuszy")

### 6a. Wnioskowanie w przód i rozwiązywanie konfliktów

Napiszcie silnik produkcyjny w cyklu dopasuj, wybierz, wykonaj. Pamięć robocza to słownik atrybut -> (wartość, znacznik czasu). Odpowiedzi ze scenariusza wpisujcie na początku jako fakty z kolejnymi znacznikami czasu. Zbiór konfliktowy tworzą reguły, których warunki są spełnione, a które jeszcze nie odpaliły (refrakcja) i których wniosek wnosi coś nowego. Zaimplementujcie trzy strategie wyboru: kolejność na liście, specyficzność (najwięcej warunków, remis rozstrzyga kolejność) i świeżość (porównanie leksykograficzne malejąco posortowanych znaczników czasu faktów użytych w warunkach, jak strategia LEX w OPS5). Silnik zatrzymuje się po pierwszej diagnozie. Zbudujcie tabelę scenariusz x strategia z diagnozą i liczbą cykli, a osobno zbiór wszystkich diagnoz, gdy silnik nie zatrzymuje się po pierwszej.

In [ ]:
# Napisz es_forward(rules, observations, strategy, stop_at_diagnosis=True) zwracające (lista diagnoz, ślad),
# gdzie ślad to lista (odpalona reguła, zbiór konfliktowy). Strategie: "kolejnosc", "specyficznosc", "swiezosc".
# Zbuduj tabelę scenariusz x strategia z pierwszą diagnozą i liczbą cykli oraz kolumną wszystkich diagnoz.

In [ ]:
# Dla scenariusza S1 wypisz dla każdej strategii kolejne cykle: odpaloną regułę i zbiór konfliktowy.

### 6b. Konsultacja wstecz z pytaniami, "dlaczego" i "jak"

Napiszcie konsultację sterowaną celem `diagnoza`: system próbuje reguł z tym wnioskiem po kolei, warunki sprawdza od lewej, a atrybut nieznany wyprowadza z innych reguł albo, gdy jest w `ASKABLE`, pyta o niego (odpowiedź ze scenariusza). Przy każdym pytaniu zapamiętajcie stos reguł, które do niego doprowadziły, i wypiszcie z niego objaśnienie "dlaczego". Po znalezieniu diagnozy wypiszcie objaśnienie "jak", czyli drzewo reguł i odpowiedzi, z których wynika. Porównajcie liczbę zadanych pytań z liczbą wszystkich obserwacji i sprawdźcie, czy kolejność prób reguł (lista albo malejąca specyficzność) zmienia diagnozę.

In [ ]:
# Napisz klasę Consultation(rules, askable, answers, order="lista" | "specyficznosc") z metodami
# value(attr, stack), run(goal="diagnoza"), why(i) (objaśnienie i-tego pytania) i how(attr) (drzewo wyprowadzenia).
# Uruchom dla scenariusza S6 z kolejnością według specyficzności, wypisz diagnozę, listę pytań,
# objaśnienie "dlaczego" dla pytania o blad_certyfikatu i objaśnienie "jak" dla diagnozy.

In [ ]:
# Dla każdego scenariusza zbuduj tabelę: diagnoza i liczba zadanych pytań przy kolejności z listy
# i przy kolejności według specyficzności.

### Wnioski do zadania 6

(uzupełnij)

## Zadanie 7: Pytania kontrolne

Odpowiedzcie na pytania kontrolne z instrukcji, po jednym do trzech zdań na pytanie.

(uzupełnij)